In [1]:
from pathlib import Path
import pandas as pd
pd.set_option('max_colwidth', 100)
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import confusion_matrix
import itertools
import scipy.stats as st
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.feature_selection import mutual_info_classif
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 500)
import rpy2
import pingouin as pg

In [2]:
# ICC - https://www.uvm.edu/~statdhtx/methods8/Supplements/icc/More%20on%20ICCs.pdf
# ICC(2,1) - Each subject is measured by each rater, and raters are considered representative
#            of a larger population of similar raters. 
#            Reliability calculated from a single measurement.

# I found two python implementations:
#     1 - pinguoin 
#     https://pingouin-stats.org/build/html/generated/pingouin.intraclass_corr.html
#     2 - R thorugh rpy2
#     https://www.r-bloggers.com/2021/06/intraclass-correlation-coefficient-in-r-quick-guide/
#     also:
#     https://stackoverflow.com/questions/40965579/intraclass-correlation-in-python-module

# ICC(2,1) for : 
# total score of HITOP
# total score of BAARS
# each subscale of HITOP
# each subscale of BAARS-IV
# GAD-7
# PHQ-8

In [3]:
#path_to_save_logs = 'logs/logs_icc/'
log_dir = Path('./log')
dat_dir = Path('../data/')
val_dir = dat_dir / 'ValSample'
fin_dir = dat_dir / 'finaldata'
path_to_save_logs = log_dir / 'logs_final/logs_icc/'
path_to_save_logs.mkdir(exist_ok=True, parents=True)

# Load data; very light processing

In [4]:
# path where data are saved
path_save_dat_gp_gridall_full = fin_dir / 'dat_gp_gridall_full.csv'
path_save_dat_en_gridall_full = fin_dir / 'dat_en_gridall_full.csv'

# load from csv
data_gp = pd.read_csv(path_save_dat_gp_gridall_full, index_col=0)
data_en = pd.read_csv(path_save_dat_en_gridall_full, index_col=0)

# create subject id column
data_gp = data_gp.rename(columns={'caseid': 'Subject'})
data_en = data_en.rename(columns={'caseid': 'Subject'})
#data_gp = data_gp.reset_index()
#data_gp = data_gp.rename(columns={'index': 'Subject'})

#data_en = data_en.reset_index()
#data_en = data_en.rename(columns={'index': 'Subject'})

# combine data
data_combined = pd.concat([data_gp, data_en])

# Functions

In [5]:
def do_icc(data, measure, show = True):
    # just the measure of interest
    data_measure = data[['Subject', measure, measure+'_recontact']]
    data_measure = data_measure.rename(columns={measure: "Original", measure+'_recontact': "Recontact"})
    data_measure_melted = pd.melt(data_measure, id_vars='Subject', value_vars=['Original','Recontact'], value_name='Score')
    data_measure_melted = data_measure_melted.rename(columns={"variable": "Session"})
    # adding Measure - don't really need to do it but eh
    shape = data_measure_melted.shape
    data_measure_melted['Measure'] = [measure] * shape[0]
    #print(measure)
    icc = pg.intraclass_corr(data=data_measure_melted, targets='Subject', raters='Session', ratings='Score').round(4)
    icc.set_index("Type")
    #print(icc)
    if show:
        plt = sns.jointplot(data=data_measure, x='Original', y='Recontact')
        plt.fig.suptitle(measure)
        #plt = plt.set(title=measure)
        #plt.subtitle(measure)
        #plt.show()
        #print(icc)
    return (icc)

def create_table (data, measures):
    df_init = pd.DataFrame()
    for measure in measures:
        if 'bothered' not in measure:
            # print(measure)
            new_icc = do_icc(data, measure, show=False)
            new_icc_row = new_icc.iloc[[1]].copy()
            new_icc_row['Measure'] = measure
            #print(new_icc_row)
            # split CI95 into two cols
            new_icc_row["lci"] = new_icc_row["CI95"].str[0]
            new_icc_row["uci"] = new_icc_row["CI95"].str[1]
            df_init = pd.concat([df_init, new_icc_row], axis = 0)
    df_result = df_init
    return (df_result)

def run_icc_analysis(data, measures):
    for measure in measures:
        if 'bothered' not in measure:
            do_icc(data, measure, show = False) # set show = "True" (default setting) if want to see the individual plots\   
    # organize results neatly
    results_table = create_table(data, measures)
    # put measure first; shift column 'Name' to first position 
    first_column = results_table.pop('Measure') 
    # insert column using insert(position,column_name,first_column) function 
    results_table.insert(0, 'Measure', first_column)
    return (results_table)

# Analysis ICC 

In [6]:
# this will be the same for any data (genpop/enriched/combined), so we can go ahead and run it
measures_for_icc = ['hitop_sum', 'baars_sum', 'phq_sum', 'gad_sum', \
                   'baars_inattention_sum', 'baars_hyperactivity_sum', 'baars_impulsivity_sum', 'baars_sct_sum',\
                   'hitop_anhedonic_depression', 'hitop_anxious_worry', 'hitop_appetite_gain',\
                   'hitop_appetite_loss', 'hitop_cognitive_problems', 'hitop_hyposomnia', 'hitop_indecisiveness',\
                   'hitop_insomnia', 'hitop_panic', 'hitop_separation_insecurity', 'hitop_shame_guilt',\
                   'hitop_situational_phobia', 'hitop_social_anxiety', 'hitop_well_being']

## 1 - Original scales (not invariant cores)

In [7]:
# -- Orig scales - genpop ---
og_icc_results_gp = run_icc_analysis(data_gp, measures_for_icc)
og_icc_results_gp.to_csv(path_to_save_logs / 'result_ICC_origscales_gp.csv', index=False)

# -- Orig scales - enriched ---
og_icc_results_en = run_icc_analysis(data_en, measures_for_icc)
og_icc_results_en.to_csv(path_to_save_logs / 'result_ICC_origscales_en.csv', index=False)

# -- Orig scales - combined ---
og_icc_results_comb = run_icc_analysis(data_combined, measures_for_icc)
og_icc_results_comb.to_csv(path_to_save_logs / 'result_ICC_origscales_combined.csv', index=False)

# --  Orig scales - combined with enriched for indecisiveness --
og_icc_results_comb.loc[og_icc_results_comb.Measure == 'hitop_indecisiveness', :] = og_icc_results_en.loc[og_icc_results_en.Measure == 'hitop_indecisiveness', :]
og_icc_results_comb.to_csv(path_to_save_logs / 'result_ICC_origscales_forplot_mostcombined_indecivinessenriched.csv', index=False)

In [8]:
og_icc_results_gp

,Measure,Type,ICC,F,df1,df2,pval,CI95,lci,uci
1,hitop_sum,"ICC(A,1)",0.9262,26.1207,397,397,0.0,"[0.91, 0.94]",0.91,0.94
1,baars_sum,"ICC(A,1)",0.8917,17.5091,397,397,0.0,"[0.87, 0.91]",0.87,0.91
1,phq_sum,"ICC(A,1)",0.8850,16.3611,397,397,0.0,"[0.86, 0.9]",0.86,0.90
1,gad_sum,"ICC(A,1)",0.8602,13.2715,397,397,0.0,"[0.83, 0.88]",0.83,0.88
1,baars_inattention_sum,"ICC(A,1)",0.8569,12.9549,397,397,0.0,"[0.83, 0.88]",0.83,0.88
1,baars_hyperactivity_sum,"ICC(A,1)",0.7961,8.8812,397,397,0.0,"[0.76, 0.83]",0.76,0.83
1,baars_impulsivity_sum,"ICC(A,1)",0.7896,8.4910,397,397,0.0,"[0.75, 0.82]",0.75,0.82
1,baars_sct_sum,"ICC(A,1)",0.8681,14.1589,397,397,0.0,"[0.84, 0.89]",0.84,0.89
1,hitop_anhedonic_depression,"ICC(A,1)",0.8893,17.1601,397,397,0.0,"[0.87, 0.91]",0.87,0.91
1,hitop_anxious_worry,"ICC(A,1)",0.8841,16.2318,397,397,0.0,"[0.86, 0.9]",0.86,0.90


In [9]:
og_icc_results_en

,Measure,Type,ICC,F,df1,df2,pval,CI95,lci,uci
1,hitop_sum,"ICC(A,1)",0.8369,11.2427,254,254,0.0,"[0.8, 0.87]",0.80,0.87
1,baars_sum,"ICC(A,1)",0.8251,10.3969,254,254,0.0,"[0.78, 0.86]",0.78,0.86
1,phq_sum,"ICC(A,1)",0.8281,10.6985,254,254,0.0,"[0.79, 0.86]",0.79,0.86
1,gad_sum,"ICC(A,1)",0.7563,7.1841,254,254,0.0,"[0.7, 0.8]",0.70,0.80
1,baars_inattention_sum,"ICC(A,1)",0.8219,10.1980,254,254,0.0,"[0.78, 0.86]",0.78,0.86
1,baars_hyperactivity_sum,"ICC(A,1)",0.7229,6.2199,254,254,0.0,"[0.66, 0.78]",0.66,0.78
1,baars_impulsivity_sum,"ICC(A,1)",0.8197,10.0702,254,254,0.0,"[0.77, 0.86]",0.77,0.86
1,baars_sct_sum,"ICC(A,1)",0.7917,8.5851,254,254,0.0,"[0.74, 0.83]",0.74,0.83
1,hitop_anhedonic_depression,"ICC(A,1)",0.8187,10.0051,254,254,0.0,"[0.77, 0.86]",0.77,0.86
1,hitop_anxious_worry,"ICC(A,1)",0.7798,8.0741,254,254,0.0,"[0.73, 0.82]",0.73,0.82


In [10]:
og_icc_results_comb

,Measure,Type,ICC,F,df1,df2,pval,CI95,lci,uci
1,hitop_sum,"ICC(A,1)",0.9310,28.0037,652,652,0.0,"[0.92, 0.94]",0.92,0.94
1,baars_sum,"ICC(A,1)",0.9002,19.0484,652,652,0.0,"[0.88, 0.91]",0.88,0.91
1,phq_sum,"ICC(A,1)",0.9068,20.4662,652,652,0.0,"[0.89, 0.92]",0.89,0.92
1,gad_sum,"ICC(A,1)",0.8733,14.7622,652,652,0.0,"[0.85, 0.89]",0.85,0.89
1,baars_inattention_sum,"ICC(A,1)",0.8836,16.1565,652,652,0.0,"[0.87, 0.9]",0.87,0.90
1,baars_hyperactivity_sum,"ICC(A,1)",0.8043,9.2722,652,652,0.0,"[0.78, 0.83]",0.78,0.83
1,baars_impulsivity_sum,"ICC(A,1)",0.8285,10.6452,652,652,0.0,"[0.8, 0.85]",0.80,0.85
1,baars_sct_sum,"ICC(A,1)",0.8799,15.6284,652,652,0.0,"[0.86, 0.9]",0.86,0.90
1,hitop_anhedonic_depression,"ICC(A,1)",0.9041,19.9053,652,652,0.0,"[0.89, 0.92]",0.89,0.92
1,hitop_anxious_worry,"ICC(A,1)",0.8981,18.6324,652,652,0.0,"[0.88, 0.91]",0.88,0.91


## 2 - Inv cores

In [11]:
#defrag datasets
data_gp = data_gp.copy()
data_en = data_en.copy()
data_combined = data_combined.copy()
# define them
for data in data_gp, data_en, data_combined:
    #defrag dataframe
    data['hitop_anhedonic_depression_invcore'] = data['hitop39'] + data['hitop77'] + data['hitop84'] + data['hitop93'] + data['hitop182'] + data['hitop230'] + data['hitop246'] 
    data['hitop_anxious_worry_invcore'] = data['hitop34'] + data['hitop89'] + data['hitop203'] + data['hitop248']
    data['hitop_appetite_gain_invcore'] = data['hitop120'] + data['hitop243'] + data['hitop275']
    data['hitop_separation_insecurity_invcore'] = data['hitop50'] + data['hitop69'] + data['hitop81'] + data['hitop136'] + data['hitop151'] + data['hitop197']
    data['hitop_situational_phobia_invcore'] = data['hitop16'] + data['hitop165'] + data['hitop225'] + data['hitop247']
    data['hitop_social_anxiety_invcore'] = data['hitop1'] + data['hitop17'] + data['hitop114'] + data['hitop129'] + data['hitop204'] + data['hitop258']
    data['hitop_well_being_invcore'] = data['hitop9'] + data['hitop23'] + data['hitop149'] + data['hitop200'] + data['hitop244'] + data['hitop250'] + data['hitop281']
    # BELOW IS JUST A PLACEHOLDER, I DIDNT DO INVARIANT CORE FOR RECONTACT
    # THIS IS JUST TO MAME THE LOOP LOOPING CORRECTLY
    data['hitop_anhedonic_depression_invcore_recontact'] = data['hitop39_recontact'] + data['hitop77_recontact'] + data['hitop84_recontact'] + data['hitop93_recontact'] + data['hitop182_recontact'] + data['hitop230_recontact'] + data['hitop246_recontact'] 
    data['hitop_anxious_worry_invcore_recontact'] = data['hitop34_recontact'] + data['hitop89_recontact'] + data['hitop203_recontact'] + data['hitop248_recontact']
    data['hitop_appetite_gain_invcore_recontact'] = data['hitop120_recontact'] + data['hitop243_recontact'] + data['hitop275_recontact']
    data['hitop_separation_insecurity_invcore_recontact'] = data['hitop50_recontact'] + data['hitop69_recontact'] + data['hitop81_recontact'] + data['hitop136_recontact'] + data['hitop151_recontact'] + data['hitop197_recontact']
    data['hitop_situational_phobia_invcore_recontact'] = data['hitop16_recontact'] + data['hitop165_recontact'] + data['hitop225_recontact'] + data['hitop247_recontact']
    data['hitop_social_anxiety_invcore_recontact'] = data['hitop1_recontact'] + data['hitop17_recontact'] + data['hitop114_recontact'] + data['hitop129_recontact'] + data['hitop204_recontact'] + data['hitop258_recontact']
    data['hitop_well_being_invcore_recontact'] = data['hitop9_recontact'] + data['hitop23_recontact'] + data['hitop149_recontact'] + data['hitop200_recontact'] + data['hitop244_recontact'] + data['hitop250_recontact'] + data['hitop281_recontact']    

In [12]:
measures_for_icc_invcores = ['hitop_anhedonic_depression_invcore', \
                             'hitop_anxious_worry_invcore', \
                             'hitop_appetite_gain_invcore',\
                             'hitop_separation_insecurity_invcore',\
                             'hitop_situational_phobia_invcore', \
                             'hitop_social_anxiety_invcore', \
                             'hitop_well_being_invcore']

In [13]:
# -- Inv scales - genpop ---
inv_core_icc_results_gp = run_icc_analysis(data_gp, measures_for_icc_invcores)
inv_core_icc_results_gp.to_csv(path_to_save_logs / 'result_ICC_invariantonly_gp.csv', index=False)

# -- Inv scales - enriched ---
inv_core_icc_results_en = run_icc_analysis(data_en, measures_for_icc_invcores)
inv_core_icc_results_en.to_csv(path_to_save_logs / 'result_ICC_invariantonly_en.csv', index=False)

# -- Inv scales - combined ---
inv_core_icc_results_comb = run_icc_analysis(data_combined, measures_for_icc_invcores)
inv_core_icc_results_comb.to_csv(path_to_save_logs / 'result_ICC_invariantonly_combined.csv', index=False)

## 3 - Results to report

In [14]:
cols_final = ['hitop_anhedonic_depression_invcore',\
              'hitop_anxious_worry_invcore',\
              'hitop_appetite_gain_invcore',\
              'hitop_appetite_loss',\
              'hitop_cognitive_problems',\
              'hitop_hyposomnia',\
              'hitop_indecisiveness',\
              'hitop_insomnia',\
              'hitop_panic',\
              'hitop_separation_insecurity_invcore',\
              'hitop_shame_guilt',\
              'hitop_situational_phobia_invcore',\
              'hitop_social_anxiety_invcore',\
              'hitop_well_being_invcore']

In [15]:
# -- final - genpop ---
to_report_icc_results_gp = run_icc_analysis(data_gp, cols_final)
to_report_icc_results_gp.to_csv(path_to_save_logs / 'result_ICC_toreport_gp.csv', index=False)

# -- final - enriched ---
to_report_icc_results_en = run_icc_analysis(data_en, cols_final)
to_report_icc_results_en.to_csv(path_to_save_logs / 'result_ICC_toreport_en.csv', index=False)

# -- final - combined ---
to_report_icc_results_comb = run_icc_analysis(data_combined, cols_final)
to_report_icc_results_comb.to_csv(path_to_save_logs / 'result_ICC_toreport_combined.csv', index=False)

